In [3]:
import pandas as pd
df=pd.read_csv('/content/IMDB Dataset.csv')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [5]:
import re
negation_words=[
    "not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"

]
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-za-z\s]"," ",text)
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [6]:
df['review']=df['review'].apply(clean_text)

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
vocab_size=20000
max_len=250
tokenizer=Tokenizer(num_words=vocab_size,oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)
X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout


model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),

    LSTM(128,dropout=0.3,recurrent_dropout=0.3)
    ,
    Dense(64,activation='relu'),
    Dropout(0.3),

    Dense(1,activation='sigmoid')
])
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
    )
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=model.fit(
    X_train_pad,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 416s 823ms/step - accuracy: 0.5694 - loss: 0.6675 - val_accuracy: 0.5800 - val_loss: 0.6627
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 416s 832ms/step - accuracy: 0.6888 - loss: 0.5785 - val_accuracy: 0.8263 - val_loss: 0.4200
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 442s 833ms/step - accuracy: 0.8691 - loss: 0.3219 - val_accuracy: 0.8817 - val_loss: 0.2802
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 417s 834ms/step - accuracy: 0.9244 - loss: 0.2034 - val_accuracy: 0.8888 - val_loss: 0.2775
Epoch 5/5
465/500 ━━━━━━━━━━━━━━━━━━━━ 27s 785ms/step - accuracy: 0.9542 - loss: 0.1341

In [11]:
loss, acc=model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 106ms/step - accuracy: 0.8894 - loss: 0.3334
Test Accuracy: 0.8894000053405762


In [12]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("Sentiment: Positive 😊")
  else:
    print("Sentiment: negative ☹️")

## Make Predictions

In [13]:
predict_sentiment("This movie was absolutely fantastic, I loved every moment of it!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 461ms/step

Review: this movie was absolutely fantastic  i loved every moment of it 
Score: 0.99065167
Sentiment: Positive 😊


In [14]:
predict_sentiment("I found this film to be quite boring and uninspired. A complete waste of time.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

Review: i found this film to be quite boring and uninspired  a complete waste of time 
Score: 0.0054447516
Sentiment: negative ☹️


In [15]:
predict_sentiment("It was an okay movie, nothing special, but not terrible either.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step

Review: it was an okay movie  nothing special  but not terrible either 
Score: 0.01088592
Sentiment: negative ☹️


In [16]:
predict_sentiment("The acting was superb, and the plot kept me on the edge of my seat throughout. Not a bad movie at all.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Review: the acting was superb  and the plot kept me on the edge of my seat throughout  not a bad movie at all 
Score: 0.98506606
Sentiment: Positive 😊
